In [3]:
!pip install scikit-learn pandas numpy matplotlib joblib

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import json
import os

In [ ]:
# Chemins sécurisés (utilise l'échantillon si les fichiers bruts sont absents)
raw_files = [
    "../data/raw/Tuesday-WorkingHours.pcap_ISCX.csv",
    "../data/raw/Wednesday-workingHours.pcap_ISCX.csv",
    "../data/raw/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
]

# Fallback vers l'échantillon si les fichiers bruts sont absents
if not os.path.exists(raw_files[0]):
    print("Fichiers bruts introuvables. Utilisation de l'échantillon de test (sample).")
    raw_files = ["../data/sample/sample_dataset.csv"]

In [ ]:
# 1.1 Chargement et nettoyage
df = pd.concat([pd.read_csv(f) for f in raw_files], ignore_index=True)
df.columns = df.columns.str.strip() # espaces en début de nom de colonne
df.replace([np.inf, -np.inf], np.nan, inplace=True) #Infinity / NaN dans Flow Bytes/s et Flow Packets/ s
df.dropna(inplace=True)

print("Distribution initiale :")
print(df["Label"].value_counts())

Distribution initiale :
Label
BENIGN              998788
DoS Hulk            230124
PortScan            158804
DoS GoldenEye        10293
FTP-Patator           7935
SSH-Patator           5897
DoS slowloris         5796
DoS Slowhttptest      5499
Heartbleed              11
Name: count, dtype: int64


In [ ]:
# 1.2 Retrait des colonnes de "triche" (Data Leakage) : ces colonnes identifient la connexion, pas le comportement — les garder ferait apprendre au modèle "cette IP = attaque" au lieu de "ce pattern = attaque" 
leak_cols = ["Flow ID", "Source IP", "Destination IP", "Source Port", "Timestamp"]
df.drop(columns=[c for c in leak_cols if c in df.columns], inplace=True)

In [8]:
# 1.3 Binarisation
df["Label_binary"] = df["Label"].apply(lambda x: "BENIGN" if x == "BENIGN" else "ATTACK")

In [9]:
# 1.4 Sélection du sous-ensemble de features "compatibles Suricata"
selected_features = [
    "Flow Duration", "Total Fwd Packets", "Total Backward Packets",
    "Total Length of Fwd Packets", "Total Length of Bwd Packets",
    "Fwd Packet Length Max", "Fwd Packet Length Mean",
    "Bwd Packet Length Max", "Bwd Packet Length Mean",
    "Flow Bytes/s", "Flow Packets/s",
    "Flow IAT Mean", "Flow IAT Std",
    "Fwd IAT Mean", "Bwd IAT Mean",
    "SYN Flag Count", "ACK Flag Count", "PSH Flag Count",
    "Average Packet Size", "Down/Up Ratio",
]

X = df[selected_features]
y_binary = df["Label_binary"]

In [11]:
# 1.5 Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.25, stratify=y_binary, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Sauvegarde des artefacts pour le modèle en direct (Phase suivante)
joblib.dump(scaler, "../models/scaler.pkl")
with open("../models/feature_columns.json", "w") as f:
    json.dump(selected_features, f)
    
print("Prétraitement terminé et artefacts sauvegardés.")

Prétraitement terminé et artefacts sauvegardés.
